# Build a QtlDataset per study

## Description

Construct a single pecotmr `QtlDataset` from one study's genotype + per-context phenotype + per-context phenotype-PC files (plus an optional uniform genotype-PC file), and serialize it to a single RDS. The resulting object is the upstream dependency for every per-gene fineMapping / TWAS / colocboost task — gene-level parallelization happens against this single object via per-gene `traitId` selectors inside the downstream pipelines.

This step does NOT iterate over genes or regions. Build the QtlDataset once per study; downstream notebooks fan out per-gene tasks against the resulting RDS.

## Inputs

- `--study` &mdash; study identifier.
- `--genotype-prefix` &mdash; PLINK2 pgen/pvar/psam prefix (no extension).
- `--phenotype CONTEXT=PATH ...` &mdash; one entry per QTL context; PATH is a bgzipped BED of phenotype values (columns `#chr`, `start`, `end`, `gene_id`, then per-sample columns).
- `--phenotype-covariates CONTEXT=PATH ...` &mdash; optional per-context molecular-trait PC TSVs (samples as rows, PCs as columns).
- `--genotype-covariates PATH` &mdash; optional TSV of genotype-derived covariates (e.g. ancestry PCs) applied uniformly across all contexts (same shape as the phenotype-PC files).
- `--maf-cutoff` / `--xvar-cutoff` &mdash; pass-through to `QtlDataset()`. These are lazy filters applied inside the accessors at fit time, not at construction.

## Output

- `{cwd}/{study}.qtl_dataset.rds`


## Example

```bash
sos run pipeline/qtl_dataset.ipynb qtl_dataset_construct \
    --cwd output \
    --study ROSMAP_DLPFC \
    --genotype-prefix input/genotype/chr22 \
    --phenotype DLPFC=input/pheno/DLPFC.bed.gz AC=input/pheno/AC.bed.gz \
    --phenotype-covariates DLPFC=input/cov/DLPFC.pcs.tsv AC=input/cov/AC.pcs.tsv \
    --genotype-covariates input/cov/genotype_pcs.tsv
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: study = str
parameter: genotype_prefix = path
# Per-context CONTEXT=PATH entries; SoS parses repeated CLI args as lists.
parameter: phenotype = []
parameter: phenotype_covariates = []
parameter: genotype_covariates = path('.')
parameter: maf_cutoff = 0.0
parameter: xvar_cutoff = 0.0
parameter: container = ''
parameter: walltime = '15m'
parameter: mem = '8G'
parameter: numThreads = 1


In [ ]:
[qtl_dataset_construct]
# Build the QtlDataset once. No fan-out; downstream notebooks load this
# RDS and parallelize per gene.
output: f"{cwd}/{study}.qtl_dataset.rds"
task: trunk_workers = 1, trunk_size = 1, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${path("code/script/pecotmr_integration/qtl_dataset_construct.R", absolute=True)} \
        --study ${study} \
        --genotype-prefix ${genotype_prefix} \
        --phenotype "${','.join(phenotype)}" \
        --phenotype-covariates "${','.join(phenotype_covariates)}" \
        --genotype-covariates ${genotype_covariates if genotype_covariates.is_file() else '""'} \
        --maf-cutoff ${maf_cutoff} \
        --xvar-cutoff ${xvar_cutoff} \
        --output ${_output}
